In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, adjusted_rand_score, normalized_mutual_info_score
from scipy.stats import mode

embeddings = pd.read_csv('embeddings.csv', header=None).values
labels = pd.read_csv('labels.csv', header=None).values.ravel()

In [2]:
print(embeddings.shape)
print(labels.shape)

(18184, 512)
(18184,)


In [3]:
n_clusters = len(np.unique(labels))

kmeans = KMeans(n_clusters=n_clusters, random_state=42)
cluster_ids = kmeans.fit_predict(embeddings)

In [4]:
def map_clusters_to_labels(cluster_ids, true_labels):
    label_mapping = {}
    for cluster in np.unique(cluster_ids):
        mask = cluster_ids == cluster
        cluster_labels = true_labels[mask]
        if len(cluster_labels) == 0:
            continue
        majority_label = mode(cluster_labels, keepdims=True).mode[0]
        label_mapping[cluster] = majority_label
    predicted_labels = np.array([label_mapping[c] for c in cluster_ids])
    return predicted_labels

In [5]:
predicted_labels = map_clusters_to_labels(cluster_ids, labels)
acc = accuracy_score(labels, predicted_labels)
ari = adjusted_rand_score(labels, cluster_ids)
nmi = normalized_mutual_info_score(labels, cluster_ids)

print(f"Cluster accuracy (via majority vote): {acc:.4f}")
print(f"Adjusted Rand Index: {ari:.4f}")
print(f"Normalized Mutual Info: {nmi:.4f}")

Cluster accuracy (via majority vote): 0.9457
Adjusted Rand Index: 0.9200
Normalized Mutual Info: 0.9838
